# ICU Mortality Prediction Model — **Phase 1**
### Sheet integration, temporal contract, and the index-admission cohort

**Overall objective.** Develop, validate, and implement an artificial intelligence model to integrate and analyze clinical, laboratory, and demographic data obtained from electronic health records and other EHI databases of ICU patients, in order to identify predictive patterns of complications.

**Primary outcome:** hospital mortality. **Secondary:** ICU mortality.

This notebook builds the **integrated analytical base**, respecting three founding rules:

1. **Temporal contract (24-h cutoff).** Only variables available *up to* the end of the first 24 hours may enter as predictors. Anything describing the rest of the hospital stay is an outcome or a leak — it stays out.
2. **Robust patient identity.** Because `EpimedCode` and `MedicalRecord` intertwine, a patient is defined by **connected components**: two admissions belong to the same person if they are linked by either key, directly or through a chain.
3. **One person, one row.** The primary cohort is the **index admission** — each patient's first hospitalization.

> **Integrity note.** The seven sheets link to each other only through the pair (`EpimedCode`, `MedicalRecord`), with no admission identifier. For most patients this pair is unique and integration is exact; for the remainder, a separate investigation showed that the sheets preserve, for each case, the same original row position — which allows the large majority of these cases to be safely recovered by positional alignment rather than excluded (§6). Only the few cases where this alignment also fails remain outside, documented for manual review.

## 1 · Load the file

> **Reproducibility note.** The original study ran in Google Colab with the source spreadsheet stored on Google Drive. In this repository, place the input file in `data/` and adjust `FILE_PATH` accordingly. No patient data is distributed with this code.

In [ ]:
# === Configuration ===
FILE_PATH = 'data/Excel01GlobalAdmission.xlsx'   # adjust to your local path

MERGE_KEYS = ['EpimedCode', 'MedicalRecord']   # key linking the seven sheets

# Output folder
OUTDIR = 'outputs/phase1'

# Policy for the few cases that EVEN AFTER recovery by positional alignment
# (Section 6) still lack a safe match — typically 4 of the original 555 ambiguous pairs.
# Recovery itself is always attempted, unconditionally; this policy only decides the fate of the residue:
#   'exclude' (recommended) -> these few residual cases stay out of the cohort
#   'keep_folha2_only'       -> they enter, but only with Folha2 data, without predictors from the other sheets
AMBIGUOUS_POLICY = 'exclude'

SHEETS = {
    'admissao':      'Folha2',   # identificação, demografia, escores, desfechos
    'comorbidades':  'Folha3',   # comorbidades e capacidade funcional
    'saps3_razoes':  'Folha4',   # razões agudas de admissão (SAPS3)
    'complic_dia1':  'Folha5',   # complicações 1ª hora / primeiras 24h
    'fisiologia6h':  'Folha6',   # 6-hour physiology and laboratory
    'recursos_uti':  'Folha7',   # >>> LEAKAGE: support over the entire hospital stay
    'diag_secund':   'Folha8',   # diagnósticos secundários (tratados na Fase 2)
}
print('Configuration loaded · ambiguous = ', AMBIGUOUS_POLICY)

In [ ]:
import os, pandas as pd, numpy as np
pd.set_option('display.max_columns', 60)

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"File not found at:\n  {FILE_PATH}\n\n"
        "Check: (1) the data file exists at the configured path; "
        "(2) the path is exact — see data/README.md for the expected layout.")
print('Arquivo encontrado:', FILE_PATH)

# The REAL header is on the 2nd row of each sheet (the 1st is a section title) -> header=1
raw = {nome: pd.read_excel(FILE_PATH, sheet_name=folha, header=1)
       for nome, folha in SHEETS.items()}

# Capture each row's original position, within its own sheet, before any transformation.
# It is this number — not the key (EpimedCode, MedicalRecord) — that Section 6 will use to
# safely recover the ambiguous pairs (see methodological note on the 555 cases).
for nome, df in raw.items():
    df['_linha_bruta'] = df.index

for nome, df in raw.items():
    print(f'{nome:14s} ({SHEETS[nome]}): {df.shape[0]:>5} linhas x {df.shape[1]:>3} colunas')

## 2 · Type cleaning and outcome derivation

Textual booleans (`Verdadeiro`/`Falso`, `Sim`/`Não` — True/False, Yes/No) become 0/1; blanks remain "not recorded" (`NaN`). Outcomes are derived from the discharge fields of `Folha2` (Sheet 2): **death (Óbito) → 1, discharge (Alta) → 0**, blank → excluded for that target.

In [ ]:
BOOL_MAP = {'verdadeiro':1,'falso':0,'sim':1,'no':0,'nao':0,'true':1,'false':0}

def to_bool(series):
    s = series.astype('string').str.strip().str.lower()
    return s.map(BOOL_MAP).astype('Float64')

def clean_booleans(df, protect):
    out = df.copy()
    for col in out.columns:
        if col in protect: continue
        vals = out[col].dropna().astype('string').str.strip().str.lower().unique()
        if len(vals) and set(vals).issubset(BOOL_MAP.keys()):
            out[col] = to_bool(out[col])
    return out

protect = set(MERGE_KEYS) | {'_linha_bruta'}
for nome in raw:
    raw[nome] = clean_booleans(raw[nome], protect)

adm = raw['admissao'].copy()
def derive_outcome(col):
    s = adm[col].astype('string').str.strip().str.lower()
    return s.map({'death':1,'obito':1,'alta':0}).astype('Float64')
adm['mortalidade_hospitalar'] = derive_outcome('HospitalDischargeName')
adm['mortalidade_uti']        = derive_outcome('UnitDischargeName')
adm['adm_datetime'] = pd.to_datetime(
    adm['UnitAdmissionDate'].astype(str) + ' ' + adm['Unit AdmissionTime'].astype(str),
    dayfirst=True, errors='coerce')

print(f'Raw admissions: {len(adm)}')
print('\nMortalidade hospitalar:')
print(adm['mortalidade_hospitalar'].value_counts(dropna=False).rename({1.0:'Death',0.0:'Discharge'}))

### 2.1 · Descriptive outcome check — HospitalDischargeName × HospitalDestinationName

`HospitalDischargeName` only distinguishes discharge/death, with no transfer category of its own. Before proceeding, we cross it with `HospitalDestinationName` to characterize up front how many admissions classified as "discharge" have destination "other hospital" — a risk of outcome misclassification bias that, without this check, would only surface after modeling.

In [ ]:
# Descriptive outcome check: HospitalDischargeName x HospitalDestinationName
# Checks whether admissions classified as "Discharge" (discharge, survivor) actually have
# destination "Other hospital" — risk of outcome misclassification bias due to transfer.
if 'HospitalDestinationName' in adm.columns:
    tab = pd.crosstab(adm['HospitalDischargeName'], adm['HospitalDestinationName'], dropna=False)
    print('Tabela cruzada: HospitalDischargeName x HospitalDestinationName')
    print(tab)

    transferidos_como_alta = adm[
        (adm['HospitalDischargeName'].astype(str).str.strip().str.lower() == 'alta') &
        (adm['HospitalDestinationName'].astype(str).str.strip() == 'Outro hospital')
    ]
    n_transf = len(transferidos_como_alta)
    print(f'\nAdmissions with HospitalDischargeName="Alta" (discharge) and destination "Outro hospital" (other hospital): '
          f'{n_transf} ({100*n_transf/len(adm):.1f}% do total)')
    print('\nWith no "transfer" category of its own in HospitalDischargeName, these cases are')
    print('coded as survivors under the current definition — outcome-bias risk to be')
    print('characterized by admission reason before fixing mortalidade_hospitalar as the primary definition.')

    tab = pd.crosstab(
        adm['HospitalDischargeName'].fillna('Not reported'),
        adm['HospitalDestinationName'].fillna('Not reported'),
        dropna=False)

    os.makedirs(OUTDIR, exist_ok=True)
    tab.to_csv(f'{OUTDIR}/checagem_desfecho_destino_fase1.csv', encoding='utf-8-sig')
    print(f'\nTabela cruzada salva em: {OUTDIR}/checagem_desfecho_destino_fase1.csv')
else:
    print('HospitalDestinationName not found in Folha2 — check not performed.')


## 3 · Remove exact duplicates (across all sheets)

Before defining the patient, we remove **true copies**: rows in `Folha2` that are fully identical — same patient, same day, same data, repeated through double entry. This differs from a genuine readmission (which has a different date); it is the same record counted twice.

Doing this *before* building `patient_id` matters: some of these copies, by repeating, created a spurious "pair ambiguity" that disappears once the duplicate is removed.

Removal is **propagated to the other sheets by original row position** (`_linha_bruta`): the same copy dropped in `Folha2` is dropped in `Folha3`–`Folha8`, so that each key remains associated with a single row in every sheet. This is necessary because, although the two copies are identical *in Folha2*, the corresponding rows in the predictor sheets may carry different clinical content — and without this propagation, the key-based `merge` would have to choose one of them silently. Propagation is performed **with verification**: before removing a position, we check that it carries the same key in that sheet; any divergence (for example, an isolated row transposition) halts removal for that position and is reported, never silenced.

In [ ]:
n_antes = len(adm)
# _linha_bruta is unique by construction (it is the original position) — it must stay out of
# the exact-duplicate check, otherwise no row would ever be considered a repeat.
cols_comparar = [c for c in adm.columns if c != '_linha_bruta']

# Identify the exact copies to remove from Folha2 (always keep the 1st occurrence).
mask_dup = adm.duplicated(subset=cols_comparar, keep='first')
pos_remover = list(adm.loc[mask_dup, '_linha_bruta'])          # posições originais descartadas
n_dup = int(mask_dup.sum())

# Key of each dropped position, to verify positional consistency across sheets.
chave_por_pos = {int(r['_linha_bruta']): (r['EpimedCode'], r['MedicalRecord'])
                 for _, r in adm.loc[mask_dup, ['_linha_bruta'] + MERGE_KEYS].iterrows()}

adm = adm.loc[~mask_dup].reset_index(drop=True)
n_depois = len(adm)

# Propagate the SAME removal (by the same positions) to the other sheets, keeping the
# 1:1 correspondence by key. Before removing, verify that the position carries,
# in that sheet, the same key as in Folha2 — safeguard against positional misalignment.
print('Exact-duplicate removal, propagated with per-position key verification:')
for nome in raw:
    if nome == 'admissao':
        print(f'  {nome:14s}: {n_dup} rows removed (Folha2, reference)')
        continue
    df = raw[nome]
    a_remover, inconsistentes = [], []
    for pos, chave in chave_por_pos.items():
        linha = df.loc[df['_linha_bruta'] == pos]
        if len(linha) == 1 and (linha['EpimedCode'].iloc[0], linha['MedicalRecord'].iloc[0]) == chave:
            a_remover.append(pos)
        else:
            inconsistentes.append(pos)
    raw[nome] = df.loc[~df['_linha_bruta'].isin(a_remover)].reset_index(drop=True)
    msg = f'  {nome:14s}: {len(a_remover)} linhas removidas'
    if inconsistentes:
        msg += f' | {len(inconsistentes)} position(s) NOT removed due to divergent key: {inconsistentes}'
    print(msg)

print(f'\nRaw admissions:              {n_antes}')
print(f'Duplicatas exatas removidas: {n_dup}')
print(f'Distinct admissions:         {n_depois}')


## 4 · Define the patient (`patient_id`)

Connected components: two admissions receive the same `patient_id` when linked by `EpimedCode` **or** by `MedicalRecord` — including through chains. This is the most conservative definition against train–test leakage.

In [ ]:
import networkx as nx   # pre-installed in Colab

def patient_id_linked(df, keys=MERGE_KEYS):
    G = nx.Graph(); G.add_nodes_from(df.index)
    for col in keys:
        for _, idx in df.groupby(col).groups.items():
            idx = list(idx)
            for a in idx[1:]:
                G.add_edge(idx[0], a)
    comp = {n:i for i, c in enumerate(nx.connected_components(G)) for n in c}
    return df.index.map(comp).astype('int64')

adm['patient_id'] = patient_id_linked(adm)

n_pac = adm['patient_id'].nunique()
adm_por_pac = adm['patient_id'].value_counts()
print(f'Unique patients (connected components): {n_pac}')
print(f'  with 1 admission:      {(adm_por_pac==1).sum()}')
print(f'  with >1 admission:     {(adm_por_pac>1).sum()}')
print(f'  maximum admissions:    {adm_por_pac.max()}')

### 4.1 · Sanity check on connected components

Connected components are vulnerable to placeholder keys (for example, a default or empty value treated as valid) that would mistakenly merge unrelated patients into a single anomalously large component. We inspect the distribution of admissions per `patient_id` and the presence of null/placeholder values in the keys.

In [ ]:
# Sanity check: an anomalously large connected component may indicate a placeholder key
# (default/empty value treated as valid) merging unrelated patients.
tam_componentes = adm['patient_id'].value_counts()
print('Distribution of admissions per patient_id:')
print(tam_componentes.describe())

LIMIAR_SUSPEITO = 10
suspeitos = tam_componentes[tam_componentes > LIMIAR_SUSPEITO]
print(f'\nPatients with more than {LIMIAR_SUSPEITO} admissions: {len(suspeitos)}')
if len(suspeitos):
    print(adm[adm['patient_id'].isin(suspeitos.index)]
          [['patient_id','EpimedCode','MedicalRecord']].drop_duplicates().to_string(index=False))

for col in MERGE_KEYS:
    vazios_ou_zero = adm[col].isna().sum() + adm[col].astype(str).str.strip().isin(['', '0']).sum()
    print(f'{col}: {vazios_ou_zero} valores nulos/placeholder')


## 5 · Index-admission cohort + integration diagnostics

We sort each `patient_id`'s admissions by date and keep the **first**. We then flag which index admissions have a **unique** (`EpimedCode`, `MedicalRecord`) pair in the database — only those link to the other sheets without ambiguity.

In [ ]:
pair_counts = adm.groupby(MERGE_KEYS).size().rename('n_par')

adm_sorted = adm.sort_values(['patient_id','adm_datetime'], kind='stable')
index_cohort = adm_sorted.drop_duplicates(subset='patient_id', keep='first').copy()
index_cohort = index_cohort.merge(pair_counts, on=MERGE_KEYS, how='left')
index_cohort['par_unico'] = index_cohort['n_par'].eq(1)

# Also flag, in the full table, which rows are index admissions and which are readmissions
adm = adm.merge(pair_counts, on=MERGE_KEYS, how='left')
adm['ambiguo'] = adm['n_par'] > 1
adm = adm.sort_values(['patient_id','adm_datetime'], kind='stable')
adm['is_indice'] = ~adm.duplicated('patient_id', keep='first')

n_total = len(index_cohort)
n_clean = int(index_cohort['par_unico'].sum())
n_ambig = n_total - n_clean
print(f'Index-admission cohort: {n_total}')
print(f'  integrable (unique pair): {n_clean}  ({100*n_clean/n_total:.1f}%)')
print(f'  ambiguous (repeated pair): {n_ambig}  ({100*n_ambig/n_total:.1f}%)')

## 6 · Apply the temporal contract and integrate the sheets

**From `Folha2`** we keep demographics, admission context, and severity scores, discarding anything after the 24-h cutoff. **`Folha3`–`Folha6`** enter as predictors. **`Folha7`** (resources used over the entire stay) is leakage and stays out. **`Folha8`** is handled in Phase 2.

**How the sheets are integrated.** The seven sheets share only the pair (`EpimedCode`, `MedicalRecord`) — there is no admission identifier. Predictor integration is done **by key**, not by row position. For the large majority of admissions this pair is unique, and the key-based `merge` associates each clinical row with the right admission unambiguously, regardless of row order across sheets.

**On ambiguous pairs.** For 555 index admissions the key repeats within `Folha2` (the same pair identifying two distinct admissions). There the key alone cannot distinguish the rows, and only for these cases do we fall back on **positional alignment** (`_linha_bruta`, the original row position captured in Section 1) as the only available disambiguation criterion. This section first attempts that recovery (§6.2) and only excludes, at the end, the few cases where alignment also fails.

### Payment source (Public/SUS, Self-pay, Private insurance)

`HealthInsuranceName` carries 40 categories — far more granularity than needed here. Instead, we derive a three-group classification plus a fourth for unreported values — without mixing assumption with data: the 526 blank records (7%) are kept apart, since the data dictionary does not clarify whether a blank represents SUS (common in systems that only name private insurers) or simply an unrecorded value.

In [ ]:
PUBLICO = ['SESA - State Department of Health']
PARTICULAR = ['Particular']

def categorizar_convenio(valor):
    if pd.isna(valor):
        return 'Not reported'
    v = str(valor).strip()
    if v in PUBLICO:
        return 'Public/SUS'
    if v in PARTICULAR:
        return 'Particular'
    return 'Private insurance'

index_cohort['OrigemPagamento'] = index_cohort['HealthInsuranceName'].apply(categorizar_convenio)
print('Distribution of OrigemPagamento (payment source) in the index-admission cohort:')
print(index_cohort['OrigemPagamento'].value_counts())
print()
print((index_cohort['OrigemPagamento'].value_counts(normalize=True)*100).round(1))

In [ ]:
FOLHA2_KEEP = [
    'EpimedCode','MedicalRecord','patient_id','_linha_bruta',
    'Age','Gender','Weight','Height','BMI',
    'AdmissionSourceName','AdmissionTypeName','AdmissionReasonName',
    'AdmissionMainDiagnosisName','Month Year','OrigemPagamento',
    'Charlson Comorbidity Index','MFI points','MFI Score','Frail Patient MFI',
    'Saps3Points','Saps3Death Probability Standard Equation','Sofa Score',
    'Status COVID 19',
    'mortalidade_hospitalar','mortalidade_uti','adm_datetime','par_unico',
]
FOLHA2_KEEP = [c for c in FOLHA2_KEEP if c in index_cohort.columns]
base_completa = index_cohort[FOLHA2_KEEP].copy()

PRED_SHEETS = ['comorbidades','saps3_razoes','complic_dia1','fisiologia6h']

# ---- 1) Unique pair: merge by key, exactly as before — safe by definition ----
clean = base_completa[base_completa['par_unico'].fillna(False)].copy()
for nome in PRED_SHEETS:
    df = raw[nome].drop_duplicates(subset=MERGE_KEYS, keep='first')   # par único -> 1:1
    novas = MERGE_KEYS + [c for c in df.columns if c not in clean.columns]
    clean = clean.merge(df[novas], on=MERGE_KEYS, how='left', validate='one_to_one')
clean['integravel'] = True
clean['recuperado_por_posicao'] = False


## 6.1 · Sheet-integration audit

The integrity of the alignment between sheets rests on two combined guarantees:

- **Exact duplicates removed consistently (§3).** True copies were dropped in *all* sheets, at the same row position and with key verification. Thus no unique-pair key is associated with more than one row in a predictor sheet.
- **Linkage by key, not by position.** For each unique-pair admission, the `merge` locates in each sheet the row whose key matches — wherever it physically sits. Ordering differences between sheets (there are, in fact, a few rows transposed between `Folha2` and the predictor sheets) do not affect which datum is associated with which patient, precisely because matching is by key.

The cell below audits the precise condition that makes this linkage correct: for **every** unique-pair admission, the key must appear exactly once in each predictor sheet. Once verified, the association is univocal — each patient receives exactly their own clinical data — and the alignment is sound.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# SHEET-INTEGRATION AUDIT
# Condition verified: for every unique-pair admission, the key
# (EpimedCode, MedicalRecord) appears exactly ONCE in each predictor sheet.
# It is this uniqueness — not row position — that makes the key-based merge
# correct and immune to ordering differences between sheets.
# ─────────────────────────────────────────────────────────────────────────
chaves_par_unico = set(zip(
    index_cohort.loc[index_cohort['par_unico'], 'EpimedCode'],
    index_cohort.loc[index_cohort['par_unico'], 'MedicalRecord']))

print('='*70)
print('SHEET-INTEGRATION AUDIT'.center(70))
print('='*70)

integra_ok = True
for nome in PRED_SHEETS:
    cont = raw[nome].groupby(MERGE_KEYS).size()
    ausentes  = [k for k in chaves_par_unico if k not in cont.index]
    repetidas = [k for k in chaves_par_unico if cont.get(k, 0) > 1]
    ok_aba = (len(ausentes) == 0) and (len(repetidas) == 0)
    integra_ok = integra_ok and ok_aba
    print(f'{nome:14s}: chave ausente = {len(ausentes):>2} | '
          f'chave repetida = {len(repetidas):>2} | '
          f'{"OK" if ok_aba else "REVISAR"}')

print('-'*70)
n_par_unico = int(index_cohort['par_unico'].sum())
merge_ok = (len(clean) == n_par_unico) and (clean['patient_id'].nunique() == n_par_unico)
print(f'Unique-pair admissions (Folha2):          {n_par_unico}')
print(f'Rows in clean after the merge:            {len(clean)}')
print(f'patient_id distintos em clean:            {clean["patient_id"].nunique()}')
print(f'  -> no duplication or row loss:          {merge_ok}')

print('-'*70)
for nome in PRED_SHEETS:
    cols_aba = [c for c in raw[nome].columns
                if c not in FOLHA2_KEEP and c != '_linha_bruta' and c in clean.columns]
    if cols_aba:
        cob = clean[cols_aba].notna().any(axis=1).mean()*100
        print(f'{nome:14s}: {len(cols_aba):>3} colunas integradas | '
              f'coverage (>=1 non-null value): {cob:5.1f}%')

# Breakdown: unique-pair admissions left with NO value from some sheet
# (all columns of that sheet empty). Not a merge failure — the key exists
# (missing key = 0 above) — but a genuine absence of the record at the source.
print('-'*70)
print('Unique-pair admissions with some sheet entirely empty:')
alguma_vazia = False
for nome in PRED_SHEETS:
    cols_aba = [c for c in raw[nome].columns
                if c not in FOLHA2_KEEP and c != '_linha_bruta' and c in clean.columns]
    if not cols_aba:
        continue
    vazias = clean[clean[cols_aba].isna().all(axis=1)]
    if len(vazias):
        alguma_vazia = True
        print(f'  {nome:14s}: {len(vazias)} admission(s) with no value from this sheet')
        for _, r in vazias[['EpimedCode','MedicalRecord']].iterrows():
            print(f'      EpimedCode={r["EpimedCode"]}  MedicalRecord={r["MedicalRecord"]}')
if not alguma_vazia:
    print('  (none — every admission has at least one value from each sheet)')
else:
    print('  -> key present in all (see section above): genuine absence of the record at')
    print('     the source, not an integration failure. Preserved as a clinical signal.')

print('='*70)
if integra_ok and merge_ok:
    print('RESULT: INTEGRATION SOUND.')
    print('Each unique-pair admission matches exactly one row of each predictor')
    print('sheet, by key (EpimedCode, MedicalRecord). Since linkage is by key and not')
    print('by position, integration is immune to ordering differences between sheets:')
    print('any isolated row transposition does not alter the clinical datum associated')
    print('with each patient.')
else:
    print('RESULT: REVIEW — see sheets flagged "REVIEW" above.')
print('='*70)

## 6.2 · Recovery of ambiguous pairs

For admissions whose key repeats within `Folha2`, the key-based `merge` cannot distinguish the two rows. Only for this subset do we resort to positional alignment: in each predictor sheet we look for the row that occupies the same original position (`_linha_bruta`) and carries the same key. Recovery is accepted only when **all** sheets contribute exactly one row under that condition; any misalignment keeps the case out, documented for manual review.

In [ ]:
# ---- 2) Ambiguous pair: attempt recovery by positional alignment ----
# Recovers only when, in ALL predictor sheets, exactly one row occupies the same
# original position as the admission — any misalignment keeps the case excluded.
ambiguas = base_completa[~base_completa['par_unico'].fillna(False)].copy()

recuperados_linhas, falhas_linhas = [], []
for _, adm_row in ambiguas.iterrows():
    linha_alvo = adm_row['_linha_bruta']
    ec, mr = adm_row['EpimedCode'], adm_row['MedicalRecord']
    novos_dados, ok = {}, True
    for nome in PRED_SHEETS:
        df = raw[nome]
        candidatos = df[(df['EpimedCode']==ec) & (df['MedicalRecord']==mr) & (df['_linha_bruta']==linha_alvo)]
        if len(candidatos) != 1:
            ok = False
            break
        linha = candidatos.iloc[0]
        for c in df.columns:
            if c not in FOLHA2_KEEP and c != '_linha_bruta':
                novos_dados[c] = linha[c]
    (recuperados_linhas if ok else falhas_linhas).append({**adm_row.to_dict(), **novos_dados} if ok else adm_row.to_dict())

recuperados = pd.DataFrame(recuperados_linhas)
ainda_ambiguos = pd.DataFrame(falhas_linhas)
if len(recuperados):
    recuperados['integravel'] = True
    recuperados['recuperado_por_posicao'] = True
if len(ainda_ambiguos):
    ainda_ambiguos['integravel'] = False
    ainda_ambiguos['recuperado_por_posicao'] = False

# ---- 3) Assemble the final base according to AMBIGUOUS_POLICY ----
# 'exclude'           -> only unique pairs + recovered cases enter (those still failing stay out)
# 'keep_folha2_only'  -> cases still failing also enter, but without predictors from the other sheets
partes = [clean, recuperados] if len(recuperados) else [clean]
if AMBIGUOUS_POLICY != 'exclude' and len(ainda_ambiguos):
    partes.append(ainda_ambiguos)
base = pd.concat(partes, ignore_index=True)

print('Integrated analytical base:', base.shape)

### Recovery results
Transparency before moving on: how many of the 555 ambiguous pairs were recovered, and how many still require manual review.

In [ ]:
n_ambiguas_total = len(ambiguas)
n_recuperados = len(recuperados)
n_ainda_ambiguos = len(ainda_ambiguos)

print('='*58)
print('RECOVERY OF AMBIGUOUS PAIRS (positional alignment)'.center(58))
print('='*58)
print(f'Total ambiguous index admissions:           {n_ambiguas_total}')
print(f'  recuperadas por alinhamento posicional:  {n_recuperados}  ({100*n_recuperados/n_ambiguas_total:.1f}%)')
print(f'  still ambiguous (manual review):         {n_ainda_ambiguos}  ({100*n_ainda_ambiguos/n_ambiguas_total:.1f}%)')
if n_recuperados:
    com_desf = recuperados['mortalidade_hospitalar'].notna().sum()
    print(f'\nOf the recovered, with known hospital outcome: {com_desf} '
          f'({recuperados["mortalidade_hospitalar"].mean()*100:.1f}% deaths)')
if n_ainda_ambiguos:
    print(f'\nPairs that remain ambiguous (key, for manual review):')
    for _, r in ainda_ambiguos[['EpimedCode','MedicalRecord']].iterrows():
        print(f'   EpimedCode={r["EpimedCode"]}  MedicalRecord={r["MedicalRecord"]}')
print('='*58)

## 7 · Complete database census

The end-to-end numerical picture — from raw admission to final cohort — for verification and for the methods section of the manuscript.

In [ ]:
print('='*58)
print('CENSO DO BANCO'.center(58))
print('='*58)
print(f'1. Raw admissions:                          {n_antes}')
print(f'2. Duplicatas exatas removidas:              {n_dup}')
print(f'3. Distinct admissions:                      {n_depois}')
print(f'4. Pacientes (componentes conexos):          {n_pac}')
print(f'     with 1 admission:                       {(adm_por_pac==1).sum()}')
print(f'     serial (>1 admission):                  {(adm_por_pac>1).sum()}')
print(f'5. Index-admission cohort:                   {n_total}')
print(f'     unique pair (integrable by key):        {n_clean}')
print(f'     ambiguous pair (repeated key):          {n_ambig}')
print(f'       -> recovered by position:               {n_recuperados}')
print(f'       -> still excluded (manual review):      {n_ainda_ambiguos}')
print(f'6. Final integrated cohort (unique pair + recovered): {n_clean + n_recuperados}')
retornos_n = n_depois - n_total
print(f'7. Returns (readmissions):                  {retornos_n}')
ret_unico = adm[(~adm["is_indice"]) & (~adm["ambiguo"])].shape[0]
ret_ambig = adm[(~adm["is_indice"]) & (adm["ambiguo"])].shape[0]
print(f'     of unique pair (integrable):            {ret_unico}')
print(f'     of ambiguous pair:                      {ret_ambig}')
print('='*58)

sem_desf = base['mortalidade_hospitalar'].isna().sum()
print(f'\nAnalytical cohort (with hospital outcome):  {len(base) - sem_desf}')
print('\nOutcome prevalence (analytical cohort):')
for alvo in ['mortalidade_hospitalar','mortalidade_uti']:
    print(f'  {alvo:24s}: {base[alvo].mean()*100:.1f}%')

In [ ]:
# Comparability: cohort used (unique pair + recovered) vs. excluded cases
# Joins the two groups that actually entered the analysis and recomputes means
# over the union — not as an average of averages.
SAPS_COL = 'Saps3Death Probability Standard Equation'

par_unico = index_cohort[index_cohort['par_unico']]
partes = [par_unico[['Age', SAPS_COL, 'mortalidade_hospitalar']]]
if n_recuperados:
    partes.append(recuperados[['Age', SAPS_COL, 'mortalidade_hospitalar']])
coorte_usada = pd.concat(partes, ignore_index=True)

def _resumo(g):
    return pd.Series({
        'n':                len(g),
        'mean_age':      g['Age'].mean(),
        'mean_SAPS3_prob': g[SAPS_COL].mean(),
        'mort_hosp_%':      100 * g['mortalidade_hospitalar'].mean(),
    })

grupos_comp = {'cohort used (unique pair + recovered)': coorte_usada}
if n_ainda_ambiguos:
    grupos_comp['excluded (still ambiguous)'] = ainda_ambiguos

comp_usada = pd.DataFrame({nome: _resumo(g) for nome, g in grupos_comp.items()})

print('=== Cohort used vs. excluded ===')
print(comp_usada.round(2))

## 8 · Mortality of the analytical cohort by period

In [ ]:
# Mortality of the analytical cohort by period — outcome stability over time.
# Unlike the readmission check (§8, which validates ambiguity), this describes
# outcome prevalence in the whole cohort and connects with the temporal validation
# (train/validation boundary = 2023).
coorte = base[base['mortalidade_hospitalar'].notna()].copy()
coorte['ano'] = pd.to_datetime(coorte['adm_datetime'], errors='coerce').dt.year

for cutoff, rotulo in [(2019, 'pre-2019 vs 2019+'),
                       (2023, 'training (≤2022) vs temporal validation (2023+)')]:
    coorte['periodo'] = np.where(coorte['ano'] < cutoff, f'< {cutoff}', f'≥ {cutoff}')
    tab = coorte.groupby('periodo')['mortalidade_hospitalar'].agg(
        n='count', mort_hosp=lambda s: 100 * s.mean()).round(2)
    print(f'=== Corte {cutoff} — {rotulo} ===')
    print(tab)
    print()

print('Cohort mortality falls from ~15.6% (training) to ~12.2% (2023+ validation):')
print('a prevalence drop to consider in Model A calibration — candidate trigger')
print('for post-hoc recalibration, without necessarily affecting discrimination.')

**Rationale for temporal validation as the primary strategy**

**Temporal validation** — training on data up to 2022 and testing on 2023 onward — was chosen as the primary evaluation strategy for Model A for reasons beyond methodological convenience.

The model's purpose is to operate prospectively: receive a patient admitted today and estimate the risk of hospital death based on first-24-hour data. This real-world use case implies the model will always be applied to future patients, admitted after the training period. Temporal validation simulates exactly this shift — the model learns from the past and is tested on the future — which validates it as the scenario closest to real clinical application. Random validation, by contrast, mixes data from all periods in the test set, underestimates the temporal generalization challenge, and tends to produce performance estimates that are overly optimistic for prospective use.

This choice also aligns with the **TRIPOD-AI** recommendations for clinical prediction models, which value validation on genuinely independent data — and future data are, by definition, the most independent possible within a single center.

## 9 · Save the bases

Four files, so nothing is lost:
- **`base_analitica_fase1.csv`** — the primary cohort (included cases, with all predictors). Starting point for Phase 2.
- **`excluidos_ambiguos_fase1.csv`** — patients with pair ambiguity whose data were aligned by row position rather than key (with 4 exceptions).
- **`coorte_indice_completa_fase1.csv`** — all index admissions with the `incluido` (included) column, as an auditable table.
- **`retornos_fase1.csv`** — readmissions, flagged by `ambiguo` (ambiguous), for the per-patient sensitivity analysis.

In [ ]:
# OUTDIR
os.makedirs(OUTDIR, exist_ok=True)

base_path = f'{OUTDIR}/base_analitica_fase1.csv'
base.to_csv(base_path, index=False, encoding='utf-8-sig')  # utf-8-sig: acentos OK no Excel

excluidos = index_cohort.loc[~index_cohort['par_unico'], FOLHA2_KEEP].copy()
excl_path = f'{OUTDIR}/excluidos_ambiguos_fase1.csv'
excluidos.to_csv(excl_path, index=False, encoding='utf-8-sig')

auditoria = index_cohort[FOLHA2_KEEP].copy()
auditoria['incluido'] = auditoria['par_unico']
audit_path = f'{OUTDIR}/coorte_indice_completa_fase1.csv'
auditoria.to_csv(audit_path, index=False, encoding='utf-8-sig')

retornos_cols = ['EpimedCode','MedicalRecord','patient_id','adm_datetime','ambiguo',
                  'Age','Saps3Points','mortalidade_hospitalar','mortalidade_uti']
retornos_cols = [c for c in retornos_cols if c in retornos.columns]
ret_path = f'{OUTDIR}/retornos_fase1.csv'
retornos[retornos_cols].to_csv(ret_path, index=False, encoding='utf-8-sig')

print(f'Included  : {base_path}  ({len(base)} rows)')
print(f'Excluded  : {excl_path}  ({len(excluidos)} rows)')
print(f'Auditoria : {audit_path}  ({len(auditoria)} linhas)')
print(f'Retornos  : {ret_path}  ({len(retornos)} linhas)')


## 10 · Preparation for Model A — hospital mortality prediction

Three safeguards, all discussed in the study design, turn the integrated base into a base *ready to learn* — without undoing anything built so far.

1. **Reserve SAPS3 as the yardstick, not as a predictor.** The SAPS3 standard-equation death probability is the *benchmark* against which the model will be compared. If it entered as a training variable, the model would merely copy SAPS3 — and the comparison would lose its meaning.
2. **Drop the nearly empty first-hour blood gas.** Columns with ~99% missingness (`Lowest pH/PaO₂/PaCO₂/FiO₂ 1h`) add weight without information; they leave at this stage.
3. **Read oncological absence as "not applicable."** In the tumor detail fields (anatomical site, malignancy type), blank means the patient does not have the disease — not missing data. We convert it into an explicit label.

At the end, the base is organized into three clean roles: **predictors** (what the model reads), **outcomes** (what it predicts), and **yardstick** (SAPS3, reserved for the fair comparison).

In [ ]:
# ---- Starting point: the Model A analytical cohort ----
# base is already the unique-pair index-admission cohort (5,148 patients).
modelo_a = base.copy()
print('Coorte do Modelo A:', modelo_a.shape)

# Drop records without the primary outcome (hospital mortality)
antes = len(modelo_a)
modelo_a = modelo_a[modelo_a['mortalidade_hospitalar'].notna()].copy()
print(f'Removed {antes - len(modelo_a)} without hospital outcome -> analytical cohort: {len(modelo_a)}')

### 10.1 · Characterization of exclusions for unknown hospital outcome

The same care applied to recovering ambiguous pairs (§7) is extended here: we compare age and SAPS3 probability between patients with and without a known `mortalidade_hospitalar` (hospital mortality), to rule out informative missingness (MNAR) before treating this exclusion as trivial.

In [ ]:
# Characterize exclusions for unknown hospital outcome — the same care
# already applied to recovering ambiguous pairs, to rule out informative missingness.
sem_desfecho = base[base['mortalidade_hospitalar'].isna()]
com_desfecho = base[base['mortalidade_hospitalar'].notna()]

comp_desfecho = pd.DataFrame({
    'with known outcome': [len(com_desfecho), com_desfecho['Age'].mean(),
                                 com_desfecho['Saps3Death Probability Standard Equation'].mean()],
    'without outcome (excluded)': [len(sem_desfecho), sem_desfecho['Age'].mean(),
                                   sem_desfecho['Saps3Death Probability Standard Equation'].mean()],
}, index=['n', 'mean_age', 'mean_SAPS3_prob'])
print(comp_desfecho.round(2))


In [ ]:
# ---- Adjustment 1 · Reserve SAPS3 as the yardstick (benchmark), outside the predictors ----
BENCHMARK_COL = 'Saps3Death Probability Standard Equation'
saps3_benchmark = modelo_a[BENCHMARK_COL].copy() if BENCHMARK_COL in modelo_a.columns else None
print('Benchmark reservado:', BENCHMARK_COL, '| n =', saps3_benchmark.notna().sum() if saps3_benchmark is not None else 0)

In [ ]:
# ---- Adjustment 2 · Drop the nearly empty first-hour blood gas ----
LIMIAR_DESCARTE = 95  # % de ausência
prop_ausente = modelo_a.isna().mean()*100
gaso_1h = ['Lowest Fi O21h','Lowest PH1h','Lowest Pa CO21h','Lowest Pa O21h','Lowest Pa O2Fi O21h']
descartar = [c for c in gaso_1h if c in modelo_a.columns and prop_ausente[c] >= LIMIAR_DESCARTE]
modelo_a = modelo_a.drop(columns=descartar)
print('Colunas de gasometria 1h descartadas (>=95% vazias):')
for c in descartar: print(f'   - {c}')

In [ ]:
# ---- Adjustment 3 · Oncological absence = "not applicable" ----
# Detail fields where blank means absence of disease (not missing data).
ONCO_TEXTO = ['Anatomic Tumor Site Name','HematologicalMalignancyTypeName']
for c in ONCO_TEXTO:
    if c in modelo_a.columns:
        modelo_a[c] = modelo_a[c].astype('string').fillna('Not applicable')

# The numeric malignancy-type code is redundant with the name and ~98% empty: drop it.
if 'HematologicalMalignancyTypeCode' in modelo_a.columns:
    modelo_a = modelo_a.drop(columns=['HematologicalMalignancyTypeCode'])

# Oncology indicators (Is...) already come as 0/1 with almost no missingness; residual blanks = 0.
onco_flags = [c for c in modelo_a.columns if c.startswith('Is') and
              any(k in c for k in ['Tumor','Malignancy','Hematological'])]
modelo_a[onco_flags] = modelo_a[onco_flags].fillna(0)
print('Oncology detail fields labeled "Not applicable":', [c for c in ONCO_TEXTO if c in modelo_a.columns])
print('Oncology indicators with blanks -> 0:', onco_flags)

In [ ]:
# ---- Organize the three roles: identifiers, outcomes, yardstick, predictors ----
IDENTIFICADORES = ['EpimedCode','MedicalRecord','patient_id','adm_datetime','par_unico','n_par',
                    '_linha_bruta','integravel','recuperado_por_posicao']
DESFECHOS       = ['mortalidade_hospitalar','mortalidade_uti']
REGUA           = [BENCHMARK_COL]

id_cols   = [c for c in IDENTIFICADORES if c in modelo_a.columns]
desf_cols = [c for c in DESFECHOS if c in modelo_a.columns]
regua_col = [c for c in REGUA if c in modelo_a.columns]

# Predictors = everything that is not identifier, outcome, or yardstick
preditores = [c for c in modelo_a.columns if c not in id_cols + desf_cols + regua_col]

X = modelo_a[preditores].copy()
y = modelo_a['mortalidade_hospitalar'].astype(int)
grupos = modelo_a['patient_id']  # for patient-level splitting (here 1 row/patient)

print(f'Predictors: {len(preditores)}  |  Outcome: mortalidade_hospitalar ({y.mean()*100:.1f}% deaths)')
print(f'Yardstick reserved: {regua_col}')
print(f'X: {X.shape}  |  y: {y.shape}')

In [ ]:
# ---- Final snapshot of the prepared base ----
print('=== Base ready for Model A ===')
print(f'Pacientes (linhas):        {len(modelo_a)}')
print(f'Preditores (colunas X):    {X.shape[1]}')
print(f'  numeric:                 {X.select_dtypes(include="number").shape[1]}')
print(f'  categorical (text):      {X.select_dtypes(include=["object","string"]).shape[1]}')
print(f'Primary outcome:           mortalidade_hospitalar ({y.sum()} deaths, {y.mean()*100:.1f}%)')

# Remaining missingness among predictors (for imputation in the next step)
aus = (X.isna().mean()*100)
aus = aus[aus>0].sort_values(ascending=False)
print(f'\nPredictors with missingness to impute in modeling: {len(aus)}')
print(aus.head(10).round(1).to_string())

In [ ]:
# ---- Save the base prepared for Model A ----
os.makedirs(OUTDIR, exist_ok=True)
modelo_a_path = f'{OUTDIR}/modelo_A_preparado.csv'
# stores predictors + outcomes + yardstick + identifiers, all together and labeled
modelo_a.to_csv(modelo_a_path, index=False, encoding='utf-8-sig')
print(f'Base do Modelo A salva: {modelo_a_path}  ({modelo_a.shape[0]} linhas x {modelo_a.shape[1]} colunas)')

### What Section 10 delivered

The Model A base now has three clear roles: the **predictors** the model will read, the **outcome** it will learn to predict, and the **reserved SAPS3** as the honest yardstick of the comparison. The empty blood-gas fields were removed; oncological absence was recoded as "not applicable."

### Next steps (Model A modeling)
- **Imputation with missingness indicators** — for predictors that still have blanks, create a "test not drawn" flag (which in the ICU is itself a clinical signal) before filling values.
- **Encode categoricals** (one-hot) and define the **two variable sets**: with ready-made scores and with raw components only.
- **Patient-level train/test split** and, as planned, a **temporal validation** using `Month Year` / `adm_datetime`.
- **Model from simple to powerful**: regularized logistic regression → gradient boosting → SHAP, always benchmarked against SAPS3 via the DeLong test.

---
### What this phase delivered
- A complete, reproducible census of the database, with exact duplicates removed before any other definition.
- `patient_id` via connected components — patient identity robust to the two intertwined keys.
- An **integrated and auditable** base, with the temporal contract applied and the index-admission cohort defined.
- The readmission profile characterized — pair ambiguity is not random; it concentrates in patients with a readmission history.
- **Positional recovery of ambiguous pairs**: of the 555 cases originally excluded, the large majority were recovered by aligning on original row position (not by key) — enlarging the final cohort without introducing leakage or requiring assumptions about which admission owns which clinical data. The few cases where alignment also fails remain documented for manual review.

### Next steps (Phase 2)
- Consolidate comorbidities into counts and groups; derive physiological ratios from the 6-hour values.
- Reduce the ~130 secondary-diagnosis fields ("present on admission" only) to something tractable — applying the **same positional-recovery logic** when joining Folha8, since it too may show isolated misalignments for the cases recovered in this phase.
- Create **missingness indicators** and the descriptive "Table 1."
- Assemble the **two variable sets**: (a) with ready-made scores; (b) raw components only.